## Data Collection 

This notebook pulls and cleans New York City Office of Administrative Trials and Hearings (OATH) data related to street vending enforcement from NYC Open Data. These functions loop through every year available (1998 - 2025) to pull all ~110,000 vending violations. The original dataframe is saved as `full_OATH_df` and a working copy is saved as `violations_df_full`. Enforcement agency names are then normalized for future analysis.

In [15]:
# Setup

import pandas as pd
import sys, os
sys.path.append(os.path.abspath("..")) 
from src.collection import fetch_data, normalize_agency_name

In [2]:
# Pulling original dataset

all_rows = []
for yr in range(1998, 2026):
    print(f"Fetching data for year {yr}")
    year_rows = fetch_data(yr)
    all_rows.extend(year_rows)

print(f"Total rows fetched: {len(all_rows)}")

original_OATH_df = pd.DataFrame(all_rows)

# Save to csv
DATA_DIR = "../data/raw"
os.makedirs(DATA_DIR, exist_ok=True)
original_OATH_df.to_csv(f"{DATA_DIR}/original_OATH_df.csv", index=False)

Fetching data for year 1998
Year 1998 offset 0: fetched 1000 records
Year 1998 offset 1000: fetched 186 records
Fetching data for year 1999
Year 1999 offset 0: fetched 1000 records
Year 1999 offset 1000: fetched 1000 records
Year 1999 offset 2000: fetched 1000 records
Year 1999 offset 3000: fetched 244 records
Fetching data for year 2000
Year 2000 offset 0: fetched 1000 records
Year 2000 offset 1000: fetched 1000 records
Year 2000 offset 2000: fetched 1000 records
Year 2000 offset 3000: fetched 1000 records
Year 2000 offset 4000: fetched 512 records
Fetching data for year 2001
Year 2001 offset 0: fetched 1000 records
Year 2001 offset 1000: fetched 1000 records
Year 2001 offset 2000: fetched 1000 records
Year 2001 offset 3000: fetched 1000 records
Year 2001 offset 4000: fetched 118 records
Fetching data for year 2002
Year 2002 offset 0: fetched 1000 records
Year 2002 offset 1000: fetched 1000 records
Year 2002 offset 2000: fetched 1000 records
Year 2002 offset 3000: fetched 1000 records

In [10]:
# Saving a sample

SAMPLE_DIR = "../data/sample"
os.makedirs(SAMPLE_DIR, exist_ok=True)
original_OATH_df.sample(100).to_csv(f"{SAMPLE_DIR}/original_OATH_df_sample.csv", index=False)

In [75]:
# Normalizing names

# Create a copy of original dataframe to work with
violations_df_full = original_OATH_df.copy()

violations_df_full['agency_normalized'] = violations_df_full['issuing_agency'].apply(normalize_agency_name)

# Check unmatched
unmatched = violations_df_full[
    violations_df_full['agency_normalized'] == violations_df_full['issuing_agency']
]['issuing_agency'].value_counts()

print("Unmatched agencies:")
print(unmatched)

print("\nNormalization summary:")
print(violations_df_full.groupby(['issuing_agency', 'agency_normalized']).size().reset_index(name='count'))

Unmatched agencies:
Series([], Name: count, dtype: int64)

Normalization summary:
                    issuing_agency              agency_normalized  count
0   AGENCY CODE MISSING OR INVALID         Missing/Invalid Agency   1621
1                     CONSUMER AFF               Consumer Affairs      9
2           COOLING TOWERS - DOHMH           Health/Mental Health      1
3                        CSTDL AGY                  Miscellaneous     69
4       DCA - TICKETSELLERBUSINESS               Consumer Affairs      7
5              DCA - Ticket Seller               Consumer Affairs      1
6    DEP - BUREAU OF CUST. SERVICE       Environmental Protection      3
7   DEP - BUREAU OF ENV. COMPLIANC       Environmental Protection     33
8        DEP - HAZARDOUS MATERIALS       Environmental Protection      1
9              DEP - RIGHT TO KNOW       Environmental Protection     57
10                     DEP. POLICE                         Police      5
11        DEPT OF CONSUMER AFFAIRS        

In [76]:
# Preparing dataset for future analysis

# Deduplicating
print("Number of Duplicates:", violations_df_full.duplicated().sum())
violations_df_full = violations_df_full.drop_duplicates()

# Dropping unnecessary columns
cols_to_keep = ['ticket_number', 'violation_date', 'violation_time', 'issuing_agency',
       'respondent_last_name', 'balance_due', 'violation_location_borough',
       'violation_location_street_name', 'respondent_address_borough', 'respondent_address_house',
       'respondent_address_zip_code', 'hearing_status', 'hearing_result',
       'scheduled_hearing_location', 'hearing_date', 'hearing_time',
       'decision_date', 'total_violation_amount', 'penalty_imposed',
       'paid_amount', 'additional_penalties_or_late_fees', 'compliance_status',
       'charge_1_code', 'charge_1_code_section', 'charge_1_code_description',
       'charge_1_infraction_amount', 'violation_location_house',
       'violation_location_city', 'violation_location_zip_code',
       'decision_location_borough', 'date_judgment_docketed',
       'charge_2_code', 'charge_2_code_section', 'charge_2_code_description',
       'charge_2_infraction_amount']
violations_df_full = violations_df_full[cols_to_keep]

# Save to csv
DATA_DIR = "../data/processed"
os.makedirs(DATA_DIR, exist_ok=True)
violations_df_full.to_csv(f"{DATA_DIR}/violations_df_full.csv", index=False)

Number of Duplicates: 147
